# 02 — Leakage-safe baseline models

This notebook explains and presents the artifacts produced by the packaged training command. It does not reimplement model fitting.

The modeling goal is to predict whether the room is occupied (`1`) or unoccupied (`0`) from environmental sensor readings. We compare several algorithms and ask a second practical question: **how much performance is lost if the Light sensor is unavailable?**

Run this first if artifacts are absent:

```powershell
$env:PYTHONPATH = "src"
python -m sensorbudget.modeling.train
```

The Python package performs training and evaluation. This notebook reads the saved metrics and predictions so that we can inspect and explain the results without retraining the models.

## Models included in the comparison

Five model types are evaluated:

- **Dummy prior:** ignores the sensors and provides a minimum benchmark. A useful model must outperform it.
- **Logistic regression:** combines weighted sensor values into an occupancy probability. It is comparatively simple and interpretable.
- **Decision tree:** learns a sequence of rules such as “if Light is above this value and CO2 is above that value.” It can represent nonlinear relationships.
- **Random forest:** averages many decision trees. This usually makes predictions more stable than using one tree.
- **Histogram gradient boosting:** builds small trees sequentially, with each new tree concentrating on earlier mistakes. It can learn complex relationships efficiently.

These are baseline configurations rather than extensively tuned models. The objective is to establish a credible comparison before performing more specialized experiments.

## Inputs and experiment structure

Every model is evaluated with two feature sets:

1. **All sensors:** Temperature, Humidity, Light, CO2, and HumidityRatio.
2. **No Light:** the same measurements with Light removed.

The target is `Occupancy`. Row IDs, timestamps, split labels, and the target itself are not model inputs.

Model comparison happens only inside the supplied training period. After selecting one model for each feature set, those two finalists are trained on the complete training period and evaluated on `test_1` and `test_2`.

In [ ]:
# Import tabular, metric, and interactive plotting tools.
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import f1_score, precision_recall_curve, precision_score, recall_score

PLOTLY_TEMPLATE = "plotly_white"
FEATURE_COLORS = {"all_sensors": "#4C78A8", "no_light": "#F58518"}

In [ ]:
# Locate the repository independently of the Jupyter launch directory.
def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate pyproject.toml")


PROJECT_ROOT = find_project_root()
ARTIFACT_DIR = PROJECT_ROOT / "models" / "baseline"

required = [
    "cv_fold_metrics.csv",
    "cv_predictions.csv",
    "cv_summary.csv",
    "heldout_metrics.csv",
    "heldout_predictions.csv",
]
missing = [name for name in required if not (ARTIFACT_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        "Run python -m sensorbudget.modeling.train; missing: " + ", ".join(missing)
    )

In [ ]:
# Load generated results; model artifacts remain outside version control.
cv_folds = pd.read_csv(
    ARTIFACT_DIR / "cv_fold_metrics.csv",
    parse_dates=["validation_start", "validation_end"],
)
cv_predictions = pd.read_csv(
    ARTIFACT_DIR / "cv_predictions.csv",
    parse_dates=["date"],
)
cv_summary = pd.read_csv(ARTIFACT_DIR / "cv_summary.csv")
heldout = pd.read_csv(ARTIFACT_DIR / "heldout_metrics.csv")
predictions = pd.read_csv(
    ARTIFACT_DIR / "heldout_predictions.csv",
    parse_dates=["date"],
)

display(
    cv_summary.sort_values(["feature_set", "f1_mean"], ascending=[True, False])[
        ["feature_set", "model", "f1_mean", "f1_std", "precision_mean", "recall_mean"]
    ]
)

## Chronological cross-validation

Cross-validation estimates how a model performs on unseen data. Because these observations form a time series, rows are not shuffled randomly. Each model learns from an earlier block and predicts a later block.

Five expanding folds are used. The training history grows from one fold to the next, while validation always occurs later in time. This avoids leaking nearly identical adjacent readings between training and validation.

The bar length below is the **mean F1 score** across the five folds. Higher is better. The error bar is one standard deviation and indicates how much performance changes between time periods. One fold is an all-unoccupied weekend, so every model has occupied-class F1 equal to zero there.

In [ ]:
# Compare candidate mean F1 within each feature configuration.
fig = make_subplots(rows=1, cols=2, subplot_titles=("All sensors", "No Light"))
for column, feature_set in enumerate(["all_sensors", "no_light"], start=1):
    subset = cv_summary.loc[cv_summary["feature_set"] == feature_set].sort_values("f1_mean")
    fig.add_trace(
        go.Bar(
            x=subset["f1_mean"],
            y=subset["model"],
            orientation="h",
            error_x={"type": "data", "array": subset["f1_std"]},
            marker_color=FEATURE_COLORS[feature_set],
            text=subset["f1_mean"].map(lambda value: f"{value:.3f}"),
            textposition="inside",
            showlegend=False,
        ),
        row=1,
        col=column,
    )
fig.update_xaxes(title_text="Mean validation F1", range=[0, 1.25])
fig.update_layout(template=PLOTLY_TEMPLATE, title="Baseline model comparison", height=500)
fig.show()

### Reading the model-comparison chart

- A longer bar indicates better average balance between occupied-class precision and recall.
- A large error bar indicates that performance is unstable across time periods.
- The dummy model's F1 is zero because it does not identify the minority occupied class at the 0.5 threshold.
- Models should be compared within the same feature panel. The no-Light panel is intentionally a harder problem.

The highest mean F1 model in each panel becomes that feature set's finalist. Mean average precision and Brier score are used only as tie-breakers.

In [ ]:
# Expose temporal variability rather than relying only on the mean.
selected_pairs = {
    ("all_sensors", "hist_gradient_boosting"),
    ("no_light", "logistic_regression"),
}
fig = go.Figure()
for feature_set, model in selected_pairs:
    subset = cv_folds.loc[
        (cv_folds["feature_set"] == feature_set) & (cv_folds["model"] == model)
    ]
    fig.add_trace(
        go.Scatter(
            x=subset["fold"],
            y=subset["f1"],
            mode="lines+markers",
            name=feature_set,
            marker_color=FEATURE_COLORS[feature_set],
            customdata=subset[["validation_occupied_rate"]],
            hovertemplate="Fold %{x}<br>F1: %{y:.3f}<br>Occupied: %{customdata[0]:.1%}<extra>%{fullData.name}</extra>",
        )
    )
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Selected models across chronological folds",
    xaxis_title="Validation fold",
    yaxis_title="F1",
)
fig.update_xaxes(dtick=1)
fig.update_yaxes(range=[0, 1.05])
fig.show()

### Reading the fold-level chart

This chart prevents the mean from hiding temporal variation. Fold 3 has no occupied observations, so occupied-class F1 is zero by definition. The important comparison is how the models behave in the other later blocks and whether their performance changes substantially over time.

After validation, histogram gradient boosting is selected for all sensors and logistic regression is selected for the no-Light configuration. Each winner is then refitted using every row in the supplied training period.

## Validation performance across classification thresholds

The models produce probabilities rather than fixed classes. A threshold converts those probabilities into occupied or unoccupied predictions. The chart below recalculates precision, recall, and F1 for thresholds from 0.01 to 0.99 using only chronological validation predictions.

- Lower thresholds usually increase recall but create more false occupied predictions.
- Higher thresholds usually increase precision but miss more occupied rows.
- The F1 peak shows the threshold with the best validation balance between precision and recall.
- The dotted line marks the current baseline threshold of 0.5.

This is the correct data for investigating a threshold because the held-out tests remain untouched.

In [ ]:
# Calculate threshold-dependent metrics from out-of-fold validation probabilities.
selected_models = {
    "all_sensors": "hist_gradient_boosting",
    "no_light": "logistic_regression",
}
threshold_grid = np.linspace(0.01, 0.99, 99)
threshold_fold_rows = []

for feature_set, model in selected_models.items():
    subset = cv_predictions.loc[
        (cv_predictions["feature_set"] == feature_set)
        & (cv_predictions["model"] == model)
    ]
    # Calculate metrics inside each time fold before averaging. This preserves
    # the same period-by-period weighting used for model selection.
    for fold, fold_data in subset.groupby("fold"):
        actual = fold_data["Occupancy"]
        probability = fold_data["probability_occupied"]
        for threshold in threshold_grid:
            predicted = (probability >= threshold).astype(int)
            threshold_fold_rows.append(
                {
                    "feature_set": feature_set,
                    "fold": fold,
                    "threshold": threshold,
                    "precision": precision_score(actual, predicted, zero_division=0),
                    "recall": recall_score(actual, predicted, zero_division=0),
                    "f1": f1_score(actual, predicted, zero_division=0),
                }
            )

threshold_fold_metrics = pd.DataFrame(threshold_fold_rows)
threshold_metrics = (
    threshold_fold_metrics.groupby(["feature_set", "threshold"], as_index=False)
    [["precision", "recall", "f1"]]
    .mean()
)
metric_colors = {"precision": "#4C78A8", "recall": "#F58518", "f1": "#54A24B"}
fig = make_subplots(rows=1, cols=2, subplot_titles=("All sensors", "No Light"))

for column, feature_set in enumerate(["all_sensors", "no_light"], start=1):
    subset = threshold_metrics.loc[threshold_metrics["feature_set"] == feature_set]
    for metric in ["precision", "recall", "f1"]:
        fig.add_trace(
            go.Scatter(
                x=subset["threshold"],
                y=subset[metric],
                mode="lines",
                name=metric.title(),
                legendgroup=metric,
                showlegend=column == 1,
                line_color=metric_colors[metric],
                customdata=np.column_stack([np.repeat(feature_set, len(subset))]),
                hovertemplate=(
                    "Feature set: %{customdata[0]}<br>"
                    "Threshold: %{x:.2f}<br>"
                    f"{metric.title()}: %{{y:.3f}}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )

    best = subset.loc[subset["f1"].idxmax()]
    fig.add_vline(
        x=0.5,
        line_dash="dot",
        line_color="black",
        annotation_text="Current: 0.5",
        row=1,
        col=column,
    )
    fig.add_trace(
        go.Scatter(
            x=[best["threshold"]],
            y=[best["f1"]],
            mode="markers",
            marker={"size": 12, "symbol": "diamond", "color": metric_colors["f1"]},
            name="Best validation F1",
            showlegend=column == 1,
            hovertemplate=(
                "Best validation F1<br>"
                "Threshold: %{x:.2f}<br>"
                "F1: %{y:.3f}<extra></extra>"
            ),
        ),
        row=1,
        col=column,
    )

fig.update_xaxes(title_text="Classification threshold", range=[0, 1])
fig.update_yaxes(title_text="Validation metric", range=[0, 1.05])
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="Mean chronological validation metrics by threshold",
    hovermode="x unified",
)
fig.show()

best_thresholds = (
    threshold_metrics.loc[threshold_metrics.groupby("feature_set")["f1"].idxmax()]
    [["feature_set", "threshold", "precision", "recall", "f1"]]
    .reset_index(drop=True)
)
display(best_thresholds)

The diamond identifies the threshold with the highest pooled out-of-fold validation F1. This is exploratory evidence, not yet the final operating threshold: a production choice should also consider the relative cost of false occupied and false unoccupied decisions.

## Held-out evaluation

Only the selected model for each feature configuration is evaluated on the two supplied test periods. These tests were not used to select the model.

The grouped bars report four threshold-dependent metrics:

- **Precision:** when the model predicts occupied, how often is it correct?
- **Recall:** of all truly occupied rows, how many does the model find?
- **F1:** a combined score that is high only when both precision and recall are reasonably high.
- **Balanced accuracy:** the average detection rate for the occupied and unoccupied classes.

All four range from 0 to 1, and higher is better. Predictions currently use the default probability threshold of 0.5.

In [ ]:
# Compare threshold metrics across test periods and feature configurations.
metric_names = ["precision", "recall", "f1", "balanced_accuracy"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"))
for column, split in enumerate(["test_1", "test_2"], start=1):
    subset = heldout.loc[heldout["split"] == split]
    for _, row in subset.iterrows():
        fig.add_trace(
            go.Bar(
                x=metric_names,
                y=[row[metric] for metric in metric_names],
                name=row["feature_set"],
                legendgroup=row["feature_set"],
                showlegend=column == 1,
                marker_color=FEATURE_COLORS[row["feature_set"]],
            ),
            row=1,
            col=column,
        )
fig.update_yaxes(title_text="Score", range=[0, 1.05])
fig.update_layout(template=PLOTLY_TEMPLATE, title="Held-out threshold metrics", barmode="group")
fig.show()

display(
    heldout[
        ["feature_set", "model", "split", "precision", "recall", "f1", "average_precision", "roc_auc", "brier_score"]
    ]
)

### Interpreting the held-out bars

The all-sensor finalist remains strong in both tests. In `test_2`, its recall is higher than its precision: it finds most occupied rows, but creates more false occupied predictions.

The no-Light finalist performs reasonably in `test_1` but deteriorates in `test_2`. Its low precision means that many rows predicted as occupied are actually empty. This supports the EDA finding that non-Light sensor distributions shift between periods.

## Precision-recall curves

A probability model can use thresholds other than 0.5. Lowering the threshold usually detects more occupied rows but also creates more false alarms. Raising it usually improves precision but misses more occupancy.

The precision-recall curve shows this trade-off across every possible threshold:

- x-axis: recall;
- y-axis: precision;
- curves closer to the upper-right corner are better;
- average precision summarizes the curve, with higher values indicating better probability ranking.

In [ ]:
# Plot precision-recall curves from saved held-out probabilities.
fig = make_subplots(rows=1, cols=2, subplot_titles=("Test 1", "Test 2"))
for column, split in enumerate(["test_1", "test_2"], start=1):
    for feature_set in ["all_sensors", "no_light"]:
        subset = predictions.loc[
            (predictions["source_split"] == split)
            & (predictions["feature_set"] == feature_set)
        ]
        precision, recall, thresholds = precision_recall_curve(
            subset["Occupancy"], subset["probability_occupied"]
        )
        # precision and recall contain one extra endpoint that has no
        # corresponding threshold, so exclude that endpoint when attaching
        # thresholds to Plotly hover data.
        fig.add_trace(
            go.Scatter(
                x=recall[:-1],
                y=precision[:-1],
                mode="lines",
                name=feature_set,
                legendgroup=feature_set,
                showlegend=column == 1,
                line_color=FEATURE_COLORS[feature_set],
                customdata=thresholds,
                hovertemplate=(
                    "Feature set: %{fullData.name}<br>"
                    "Recall: %{x:.3f}<br>"
                    "Precision: %{y:.3f}<br>"
                    "Threshold: %{customdata:.3f}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )

        # Highlight the currently used 0.5 classification threshold.
        current_prediction = (subset["probability_occupied"] >= 0.5).astype(int)
        current_precision = precision_score(
            subset["Occupancy"], current_prediction, zero_division=0
        )
        current_recall = recall_score(
            subset["Occupancy"], current_prediction, zero_division=0
        )
        fig.add_trace(
            go.Scatter(
                x=[current_recall],
                y=[current_precision],
                mode="markers",
                marker={
                    "size": 11,
                    "symbol": "diamond",
                    "color": FEATURE_COLORS[feature_set],
                    "line": {"color": "black", "width": 1},
                },
                customdata=[0.5],
                name=f"{feature_set} at 0.5",
                legendgroup=feature_set,
                showlegend=False,
                hovertemplate=(
                    "Current operating point<br>"
                    "Feature set: %{fullData.name}<br>"
                    "Recall: %{x:.3f}<br>"
                    "Precision: %{y:.3f}<br>"
                    "Threshold: %{customdata:.3f}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=column,
        )
fig.update_xaxes(title_text="Recall", range=[0, 1])
fig.update_yaxes(title_text="Precision", range=[0, 1.05])
fig.update_layout(template=PLOTLY_TEMPLATE, title="Held-out precision-recall curves")
fig.show()

### Reading the precision-recall curves

The all-sensor curve stays high across a broad recall range in both tests, showing that its strong result is not limited to the 0.5 threshold. The no-Light curve is weaker, especially in `test_2`, which means threshold adjustment alone cannot completely recover the lost information.

The held-out curves are descriptive final evaluation results. We must not choose a new threshold from them. Threshold selection should use validation predictions from the training period.

## Conclusions

The experiment supports the following conclusions:

- **Best all-sensor baseline:** histogram gradient boosting has the highest mean chronological validation F1. At the unchanged 0.5 threshold, held-out F1 is approximately 0.930 on `test_1` and 0.883 on `test_2`.
- **Best no-Light baseline:** logistic regression is the strongest no-Light candidate by mean chronological F1. It reaches held-out F1 of approximately 0.832 on `test_1`, but falls to 0.546 on `test_2`.
- **Value of Light:** removing Light causes a clear and period-dependent performance loss. The no-Light model produces many false occupied predictions in `test_2`, showing that the remaining environmental relationships are less stable across periods.
- **Temporal variability:** model performance changes substantially between chronological folds. The all-unoccupied weekend fold cannot be assessed with occupied-class F1 alone and should also be examined through false-positive and specificity measures.
- **Threshold exploration:** validation results suggest that F1 peaks near 0.91 for the all-sensor finalist and 0.88 for the no-Light finalist. These values come from chronological validation only; the held-out results above still use 0.5.
- **Threshold decision:** the F1-optimal threshold is not automatically the best operational threshold. The final choice should also consider calibration and the relative cost of unnecessary energy use versus missed occupancy.
- **Next experiment:** compare additional sensor subsets, quantify the performance-versus-cost frontier, and then evaluate robustness to missing or faulty sensors.

Overall, the all-sensor model is a strong baseline for this dataset, while the no-Light results demonstrate why sensor availability and cross-period robustness must be treated as explicit design decisions. These results describe one room over a short collection period and do not yet establish generalization to other buildings.